In [1]:
import requests
import pandas as pd
import numpy as np

In [2]:
def get_user_id(name):
    num_results = 1
    try:
        page = requests.get(f'https://app.universaltennis.com/api/v2/search/players?query={name}&top={num_results}')
        data = page.json()
        # extract user id from either 'hits' or 'Hits' because utr has two JSON responses
        key = 'hits' if 'hits' in data else 'Hits'
        return data[key][0]['id'] if key == 'hits' else data[key][0]['Id']
    except (KeyError, IndexError, requests.exceptions.RequestException) as e:
        print(f'Error: Could not find user ID for {name}')

In [3]:
get_user_id('Sebastian Nothaft')

'2260216'

In [4]:
get_user_id('Roger Federer')

'3100657'

In [5]:
def get_utr(name):
    user_id = get_user_id(name)
    # get player utr
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}'
    page = requests.get(url)
    data = page.json()
    singles_utr = data['singlesUtr']
    return singles_utr

In [6]:
print(get_utr('Novak Djokovic'))
print(get_utr('Spencer Johnson'))

15.91
13.68


In [7]:
ucla_players = ['Gianluca Ballotta',
                'Cassius Chinlund',
                'Andrei Crabel',
                'Spencer Johnson',
                'Rudy Quan',
                'Andy Nguyen',
                'Bengt Reinhard',
                'Will Steinberg'
                'Giacomo Revelli',
                'Aadarsh Tripathi',
                'Emon van Loben Sels',
                'Leo von Bismarck',
                
                'Olivia Center',
                'Kate Fakih',
                'Bianca Fernandez',
                'Ahmani Guichard',
                'Kimberly Hance',
                'Mia Jovic',
                'Anne-Christine Lutkemeyer',
                'Elise Wagle']

In [8]:
# get UTRs of all ucla players and store in dictionary
ucla_utr_dict = {}
for player in ucla_players:
    print(player)
    utr = get_utr(player)
    ucla_utr_dict[player] = utr

Gianluca Ballotta
Cassius Chinlund
Andrei Crabel
Spencer Johnson
Rudy Quan
Andy Nguyen
Bengt Reinhard
Will SteinbergGiacomo Revelli
Aadarsh Tripathi
Emon van Loben Sels
Leo von Bismarck
Olivia Center
Kate Fakih
Bianca Fernandez
Ahmani Guichard
Kimberly Hance
Mia Jovic
Anne-Christine Lutkemeyer
Elise Wagle


In [9]:
ucla_utr_dict

{'Gianluca Ballotta': 12.0,
 'Cassius Chinlund': 13.0,
 'Andrei Crabel': 11.0,
 'Spencer Johnson': 13.68,
 'Rudy Quan': 13.54,
 'Andy Nguyen': 13.0,
 'Bengt Reinhard': 12.35,
 'Will SteinbergGiacomo Revelli': 12.0,
 'Aadarsh Tripathi': 13.01,
 'Emon van Loben Sels': 13.4,
 'Leo von Bismarck': 0.0,
 'Olivia Center': 10.01,
 'Kate Fakih': 10.11,
 'Bianca Fernandez': 9.92,
 'Ahmani Guichard': 0.0,
 'Kimberly Hance': 0.0,
 'Mia Jovic': 7.0,
 'Anne-Christine Lutkemeyer': 10.89,
 'Elise Wagle': 0.0}

## Getting Dataframe of Past Matches

#### getting singles results

player 1 is winner

In [10]:
# Convert UTR scoring into a string
def get_score_string(score):
    sets = []
    for i in range(1, 4):
        if str(i) in score:
            loser_score = score[str(i)]['loser'] if score[str(i)]['loser'] is not None else '0' # account for 0 stored as None in the UTR data
            set_score = f"{score[str(i)]['winner']}-{loser_score}"
            if score[str(i)]['tiebreak'] != None:
                set_score += f"({score[str(i)]['tiebreak']})"
            sets.append(set_score)
    return ', '.join(sets)

In [11]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [12]:
# Convert date into a more readable format
def convert_date(date):
    utc = datetime.fromisoformat(date).replace(tzinfo=ZoneInfo('UTC'))
    pst = utc.astimezone(ZoneInfo('America/Los_Angeles'))
    return pst.strftime('%Y-%m-%d')

In [13]:
def get_singles_results(name):
    user_id = get_user_id(name)

    # add ?type=singles for singles only
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}/results?type=singles'
    page = requests.get(url)
    data = page.json()

    results = []
    for event in data['events']:
        event_name = event['name']

        if len(event['draws']) == 0:
            for result in event['results']:
                date = convert_date(result['date'])
                player1 = result['players']['winner1']['firstName'] + ' ' + result['players']['winner1']['lastName']
                player2 = result['players']['loser1']['firstName'] + ' ' + result['players']['loser1']['lastName']
                player1_singles_utr = result['players']['winner1']['singlesUtr']
                player2_singles_utr = result['players']['loser1']['singlesUtr']
                score = get_score_string(result['score'])
                results.append([event_name, date, player1, player1_singles_utr, player2, player2_singles_utr, score])
        else:
            for result in event['draws'][0]['results']:
                date = convert_date(result['date'])
                player1 = result['players']['winner1']['firstName'] + ' ' + result['players']['winner1']['lastName']
                player2 = result['players']['loser1']['firstName'] + ' ' + result['players']['loser1']['lastName']
                player1_singles_utr = result['players']['winner1']['singlesUtr']
                player2_singles_utr = result['players']['loser1']['singlesUtr']
                score = get_score_string(result['score'])
                results.append([event_name, date, player1, player1_singles_utr, player2, player2_singles_utr, score])

    singles_results = pd.DataFrame(results, columns=['Event Name', 'Date', 'Player1', 'Player1 UTR', 'Player2', 'Player2 UTR', 'Score'])
    return singles_results

In [14]:
pd.set_option('display.max_rows', None)

In [15]:
singles = get_singles_results('Spencer Johnson')
singles

,Event Name,Date,Player1,Player1 UTR,Player2,Player2 UTR,Score
0,Dual Match: University of San Diego vs Univers...,2026-05-01,Oliver Tarvet,14.47,Spencer Johnson,13.68,"6-4, 7-5"
1,Dual Match: UCLA vs Arizona State University,2026-04-30,Spencer Johnson,13.68,Ofek Shimanov,13.16,"6-1, 6-4"
2,"Dual Match: University of California, Los Ange...",2026-04-30,Spencer Johnson,13.68,Mathis Bondaz,13.10,"6-1, 6-4"
3,Dual Match: Michigan State University vs Unive...,2026-04-23,Aristotelis Thanos,13.85,Spencer Johnson,13.68,"6-3, 7-5"
4,"Dual Match: University of California, Los Ange...",2026-04-22,Branko Djuric,13.60,Spencer Johnson,13.68,"6-4, 6-7(7), 1-2"
5,Dual Match: Ohio State University vs Universit...,2026-04-03,Aidan Kim,13.76,Spencer Johnson,13.68,"6-3, 6-1"
6,Dual Match: Michigan State University vs Unive...,2026-03-28,Aristotelis Thanos,13.85,Spencer Johnson,13.68,"1-6, 6-2, 6-2"
7,Dual Match: University of California Los Angel...,2026-03-26,Max Dahlin,13.89,Spencer Johnson,13.68,"6-4, 6-2"
8,"Dual Match: University of California, Los Ange...",2026-03-19,Spencer Johnson,13.68,Michael Minasyan,13.11,"6-3, 6-2"
9,"Dual Match: University of California, Los Ange...",2026-03-12,Spencer Johnson,13.68,Branko Djuric,13.60,"4-6, 6-2, 7-6(6)"


## Trying to get historical UTR

In [16]:
import pprint

In [17]:
get_user_id('Spencer Johnson')

'2580878'

In [18]:
# Spencer Johnson statistics
url = f'https://app.universaltennis.com/api/v1/player/2580878/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
page = requests.get(url)
spencer_stats = page.json()

In [19]:
pprint.pp(spencer_stats)

{'title': 'verified singles stats - Last 12 Months',
 'subtitle': 'May 2025 - May 2026',
 'monthsCount': 12,
 'currentRating': 13.68,
 'currentRatingDisplay': '13.68',
 'ratingCircle': {'singlesUtr': 13.68,
                  'ratingStatusSingles': 'Rated',
                  'ratingProgressSingles': 100.0,
                  'singlesUtrDisplay': '13.68'},
 'winsCount': 17,
 'winStreak': 8,
 'lossesCount': 10,
 'recordWinPercentage': '63%',
 'notableMatch': {'winner': {'isWinner': True,
                             'set1': 6,
                             'set2': 7,
                             'set3': 0,
                             'set4': 0,
                             'set5': 0,
                             'set6': None,
                             'tiebreakerSet1': None,
                             'tiebreakerSet2': None,
                             'tiebreakerSet3': 0,
                             'tiebreakerSet4': 0,
                             'tiebreakerSet5': 0,
            

In [20]:
# UTR history for each week
spencer_stats['extendedRatingProfile']['history']

[{'id': 577365485,
  'ratingStatus': 3,
  'rating': 13.68,
  'ratingDisplay': '13.68',
  'type': 'WeeklyAverage_Singles',
  'date': '2026-05-11T00:00:00'},
 {'id': 576115911,
  'ratingStatus': 3,
  'rating': 13.68,
  'ratingDisplay': '13.68',
  'type': 'WeeklyAverage_Singles',
  'date': '2026-05-04T00:00:00'},
 {'id': 575398910,
  'ratingStatus': 3,
  'rating': 13.68,
  'ratingDisplay': '13.68',
  'type': 'WeeklyAverage_Singles',
  'date': '2026-04-27T00:00:00'},
 {'id': 574021433,
  'ratingStatus': 3,
  'rating': 13.66,
  'ratingDisplay': '13.66',
  'type': 'WeeklyAverage_Singles',
  'date': '2026-04-20T00:00:00'},
 {'id': 572837926,
  'ratingStatus': 3,
  'rating': 13.67,
  'ratingDisplay': '13.67',
  'type': 'WeeklyAverage_Singles',
  'date': '2026-04-13T00:00:00'},
 {'id': 571171044,
  'ratingStatus': 3,
  'rating': 13.73,
  'ratingDisplay': '13.73',
  'type': 'WeeklyAverage_Singles',
  'date': '2026-04-06T00:00:00'},
 {'id': 570170494,
  'ratingStatus': 3,
  'rating': 13.82,
  'ra

In [21]:
spencer_stats['extendedRatingProfile']['history'][0]

{'id': 577365485,
 'ratingStatus': 3,
 'rating': 13.68,
 'ratingDisplay': '13.68',
 'type': 'WeeklyAverage_Singles',
 'date': '2026-05-11T00:00:00'}

In [22]:
def create_UTR_date_dict(utr_history):
    utr_date_dict = {}
    for week in utr_history:
        date = datetime.strptime(week['date'], '%Y-%m-%dT%H:%M:%S').date()
        rating = week['ratingDisplay']
        utr_date_dict[date] = rating
    return utr_date_dict

In [23]:
utr_dict = create_UTR_date_dict(spencer_stats['extendedRatingProfile']['history'])
utr_dict

{datetime.date(2026, 5, 11): '13.68',
 datetime.date(2026, 5, 4): '13.68',
 datetime.date(2026, 4, 27): '13.68',
 datetime.date(2026, 4, 20): '13.66',
 datetime.date(2026, 4, 13): '13.67',
 datetime.date(2026, 4, 6): '13.73',
 datetime.date(2026, 3, 30): '13.82',
 datetime.date(2026, 3, 23): '13.81',
 datetime.date(2026, 3, 16): '13.77',
 datetime.date(2026, 3, 9): '13.76',
 datetime.date(2026, 3, 2): '13.74',
 datetime.date(2026, 2, 23): '13.73',
 datetime.date(2026, 2, 16): '13.69',
 datetime.date(2026, 2, 9): '13.68',
 datetime.date(2026, 2, 2): '13.69',
 datetime.date(2026, 1, 26): '13.72',
 datetime.date(2026, 1, 19): '13.72',
 datetime.date(2026, 1, 12): '13.72',
 datetime.date(2026, 1, 5): '13.73',
 datetime.date(2025, 12, 29): '13.73',
 datetime.date(2025, 12, 22): '13.73',
 datetime.date(2025, 12, 15): '13.74',
 datetime.date(2025, 12, 8): '13.74',
 datetime.date(2025, 12, 1): '13.75',
 datetime.date(2025, 11, 24): '13.77',
 datetime.date(2025, 11, 17): '13.78',
 datetime.date

In [24]:
# get historical UTR at time of match, for a specific player
# match the event date to the closest date in the UTR history
def get_historical_utr(name, date):
    user_id = get_user_id(name)
    
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
    page = requests.get(url)
    stats = page.json()

    utr_history = stats['extendedRatingProfile']['history']
    utr_dict = create_UTR_date_dict(utr_history)

In [25]:
def get_closest_date(utr_dict, date):
    closest_date = None
    closest_diff = None
    for key in utr_dict.keys():
        diff = abs((key - date).days)
        if closest_diff is None or diff < closest_diff:
            closest_diff = diff
            closest_date = key
    return closest_date

In [26]:
closest = get_closest_date(utr_dict, datetime.strptime('2024-10-01', '%Y-%m-%d').date())
closest

datetime.date(2024, 9, 30)

In [27]:
utr_dict[closest]

'13.40'

bro idk

In [28]:
name = 'Spencer Johnson'
user_id = get_user_id(name)

url_stats = f'https://app.universaltennis.com/api/v1/player/{user_id}/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
page = requests.get(url_stats)
stats = page.json()

url_singles = f'https://app.universaltennis.com/api/v1/player/{user_id}/results?type=singles'
page = requests.get(url_singles)
singles = page.json()

In [29]:
pprint.pp(stats)

{'title': 'verified singles stats - Last 12 Months',
 'subtitle': 'May 2025 - May 2026',
 'monthsCount': 12,
 'currentRating': 13.68,
 'currentRatingDisplay': '13.68',
 'ratingCircle': {'singlesUtr': 13.68,
                  'ratingStatusSingles': 'Rated',
                  'ratingProgressSingles': 100.0,
                  'singlesUtrDisplay': '13.68'},
 'winsCount': 17,
 'winStreak': 8,
 'lossesCount': 10,
 'recordWinPercentage': '63%',
 'notableMatch': {'winner': {'isWinner': True,
                             'set1': 6,
                             'set2': 7,
                             'set3': 0,
                             'set4': 0,
                             'set5': 0,
                             'set6': None,
                             'tiebreakerSet1': None,
                             'tiebreakerSet2': None,
                             'tiebreakerSet3': 0,
                             'tiebreakerSet4': 0,
                             'tiebreakerSet5': 0,
            

In [30]:
stats['extendedRatingProfile'] == None

False

In [31]:
import unicodedata
def remove_accents(string):
    nfkd_form = unicodedata.normalize('NFKD', string)
    return u"".join([c for c in nfkd_form if not unicodedata.combining(c)])

In [32]:
remove_accents('Théo PAPAMALAMIS')

'Theo PAPAMALAMIS'

In [33]:
dict = {
    'firstName': 'Nicolas '
                'Ian',
    'lastName': 'Kotzen',
}

In [34]:
dict['firstName'].split()[0]

'Nicolas'

In [35]:
# if player1 == 'Spencer Johnson' then assign player1 historical utr to the utr

In [36]:
def get_historical_UTR(name, match_date):
    print(name)
    # create dicitonary of (weekly) dates and UTR ratings
    user_id = get_user_id(name)
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
    page = requests.get(url)
    stats = page.json()

    # for now, if there's no extendedRatingProfile history, return nan
    try:
        utr_history = stats['extendedRatingProfile']['history']
    except (KeyError, IndexError, TypeError):
        print(f'Error: Could not find UTR history for {name}')
        return np.nan

    utr_dict = {}
    for week in utr_history:
        date = datetime.strptime(week['date'], '%Y-%m-%dT%H:%M:%S').date()
        rating = week['ratingDisplay']
        utr_dict[date] = rating

    # get the UTR rating from the closest date
    closest_date = None
    closest_diff = None
    for key in utr_dict.keys():
        diff = abs((key - match_date).days)
        if closest_diff is None or diff < closest_diff:
            closest_diff = diff
            closest_date = key

    return utr_dict[closest_date]

In [37]:
name = 'Alex Cairo'
date = datetime.strptime('2025-04-25', '%Y-%m-%d').date()
user_id = get_user_id(name)
url = f'https://app.universaltennis.com/api/v1/player/{user_id}/stats?type=singles&resultType=verified&Months=12&fetchAllResults=false'
page = requests.get(url)
stats = page.json()

In [38]:
try:
    utr_history = stats['extendedRatingProfile']['history']
except (KeyError, IndexError, TypeError):
    print(f'Error: Could not find UTR history for {name}')

Error: Could not find UTR history for Alex Cairo


In [39]:
utr_history == None

NameError: name 'utr_history' is not defined

In [ ]:
get_historical_UTR('Alex Cairo', datetime.strptime('2025-04-25', '%Y-%m-%d').date())

Alex Cairo


TypeError: 'NoneType' object is not iterable